In [3]:
import torch
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam

In [4]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).long()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [ ]:
def train_model_reg(features, targets, model, kf, batch, epochs, optmizer, criterion):
    accuracy_score = []
    f1_score = []
    for i, (train_index, test_index) in enumerate(kf.split(features)):  #split
        print(f"Treinando modelo ----- Fold {i + 1}")
        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size=batch)
        for i_epochs in range(epochs):    
            model.train()
            for x, y in train_loader:
                optmizer.zero_grad()
                output = model(x)
                loss = criterion(output, y)
                loss.backward()
                optmizer.step()

        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(features[test_index]).float()
            y_true = targets[test_index]
            
            # output / argmax
            # result = model(x_test_tensor).cpu().numpy()
            # r2_s = r2_score(y_true, result)
            # rmse_s = root_mean_squared_error(y_true, result)
            # mse_s = mean_squared_error(y_true, result)
            # r2_scores.append(r2_s)
            # rmse_scores.append(rmse_s)
            # mse_scores.append(mse_s)
            # print(f"R^2: {r2_s}")
            # print(f"RMSE: {rmse_s}")
            # print(f"MSE: {mse_s}")

    print(f"\nResultado modelo ----- R^2: {np.mean(r2_scores)}, RMSE: {np.mean(rmse_scores)}, MSE: {np.mean(mse_scores)}")

In [ ]:
dataset = load_iris()
features = dataset.data
targets = np.expand_dims(dataset.target, axis=1)

In [5]:
model = Sequential(
    Linear(10, 32),
    ReLU(),
    Linear(32,16),
    ReLU(),
    Linear(16, 1)
)

In [6]:
optmizer = Adam(model.parameters(), lr=0.001)
criterion = MSELoss()
max_epochs = 500
batch = 12
kf = KFold(n_splits=5, shuffle=True)

In [7]:
train_model_reg(features, targets, model, kf, batch, max_epochs, optmizer, criterion)

Treinando modelo ----- Fold 1
R^2: 0.43834950490850666
RMSE: 57.18530583247017
MSE: 3270.1592031531472
Treinando modelo ----- Fold 2
R^2: 0.5283060776225577
RMSE: 53.847871222465145
MSE: 2899.59323519119
Treinando modelo ----- Fold 3
R^2: 0.3035412658028963
RMSE: 59.33452275822513
MSE: 3520.5855909463357
Treinando modelo ----- Fold 4
R^2: 0.6516457517398295
RMSE: 50.64494602424924
MSE: 2564.910557799119
Treinando modelo ----- Fold 5
R^2: 0.4340141286583773
RMSE: 53.57051229030483
MSE: 2869.799787045701

Resultado modelo ----- R^2: 0.47117134574643355, RMSE: 54.916631625542905, MSE: 3025.009674827099
